# Ozzy Ad Studio · Colab

Video (**MiniMax H3 Ref2VA FP8**) ve görsel (**Qwen Image 2.1 Turbo**) üretim sunucusunu açar. Hücreleri sırayla çalıştırın (veya *Çalışma zamanı → Tümünü çalıştır*).

**Önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir →* **G4** GPU.
**Colab Secrets** (soldaki 🔑), bu notebook'a erişim açık olmalı:
- `NGROK_AUTHTOKEN`, `NGROK_DOMAIN` (sabit adres)
- `STUDIO_TOKEN` (en az 16 karakter; yoksa bir kez üretilir ve `MyDrive/OzzyAdStudio/.studio_token` dosyasına yazılır)
- `HF_TOKEN` (isteğe bağlı)

Kod Drive'da (`MyDrive/OzzyAdStudio/code`) çalışır; ilk hücre onu GitHub ile karşılaştırıp günceller. Modeller yalnızca Drive'da durur (`MyDrive/OzzyVision-Lab/models`), bir kez indirilir; hücre 3'te `MODEL_CACHE` ile o oturum için yerel SSD'ye kopyalanabilir.

In [ ]:
#@title 0 · GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
#@title 1 · Drive ve kod (Drive'daki kopya GitHub ile karşılaştırılır)
BRANCH = "main"  #@param {type:"string"}
import os, sys, subprocess, importlib
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/drive/MyDrive/OzzyAdStudio/code'
if not os.path.isdir(CODE + '/.git'):
    os.makedirs(os.path.dirname(CODE), exist_ok=True)
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, 'https://github.com/OzzyD07/ozzy-ad-studio.git', CODE], check=True)
if CODE + '/studio-mcp' not in sys.path:
    sys.path.insert(0, CODE + '/studio-mcp')
from studio import colab
colab.sync_code(BRANCH)
colab = importlib.reload(colab)  # güncellenen kurulum kodu bu oturumda da kullanılsın

In [ ]:
#@title 2 · ComfyUI ve bağımlılıklar
%cd /content
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git
else:
    !git -C /content/ComfyUI pull -q
!pip install -q -r /content/ComfyUI/requirements.txt
!pip install -q -r {CODE}/studio-mcp/requirements.txt pyngrok huggingface_hub
!apt-get -qq install -y ffmpeg > /dev/null
!git -C /content/ComfyUI log -1 --format='ComfyUI %h (%cd)' --date=short

In [ ]:
#@title 3 · Modeller (Drive'da; eksikler indirilir, boyutlar doğrulanır)
QWEN_VARIANT = "bf16"  #@param ["bf16", "int8"]
VERIFY_SHA = False  #@param {type:"boolean"}
# drive: doğrudan Drive'dan okunur (bekleme yok, ilk yükleme ve model değişimleri yavaş)
# minimax: MiniMax (~42 GB) bu oturum için yerel SSD'ye kopyalanır · all: Qwen de (~75 GB)
MODEL_CACHE = "drive"  #@param ["drive", "minimax", "all"]
colab.ensure_models(QWEN_VARIANT, VERIFY_SHA, cache=MODEL_CACHE)

In [ ]:
#@title 4 · Başlat: ComfyUI + Studio MCP + ngrok
token, token_source = colab.studio_token()
assert colab.start_comfy(), 'ComfyUI açılmadı'
assert colab.start_server(token), 'Studio MCP açılmadı'
public_url = colab.start_tunnel()
print(f'\n🤖 MCP adresi: {public_url}/mcp')
print(f'🔑 Jeton: {token_source}  (OtoReklam: VIDEO_MCP_TOKEN ortam değişkeni)')

In [ ]:
#@title 5 · Durum ve loglar
!curl -s localhost:8000/health; echo
!echo '--- studio.log'; tail -n 15 /content/logs/studio.log
!echo '--- comfyui.log'; tail -n 15 /content/logs/comfyui.log